# Lab 4b: Tool Calling

How do LLMs interact with the outside world? Through **tool calling** — the model decides *which* function to invoke and *what arguments* to pass, then your code executes it and feeds the result back.

0. **Setup** — imports, client, load tools
1. **The Tools** — 4 simple Python functions for a travel assistant
2. **Manual Tool Calling** — make the model output JSON, parse it yourself (fragile)
3. **API-Level Tool Calling** — let the API handle structured tool calls (robust)
4. **The Tool-Calling Loop** — multi-step queries with automatic dispatch
5. **When to Answer vs. When to Call Another Tool** — the model's decision-making
6. **Tool Naming and Documentation Matter** — good vs bad tool definitions
7. **Asking Clarification Questions** — the model's most flexible "tool"

Run cells in order.

## 0. Setup

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
MODEL_STRONG = "gpt-5.2"
print(f"Client ready — default model: {MODEL}")

In [ ]:
from tools import TOOL_REGISTRY, get_openai_tool_schemas, execute_tool

print(f"Loaded {len(TOOL_REGISTRY)} tools: {list(TOOL_REGISTRY.keys())}")

## 1. The Tools

Our travel assistant has 4 tools — plain Python functions returning fake data.
The interesting part is not the tools themselves, but how we connect them to the LLM.

| Tool | Purpose | Key parameters |
|------|---------|----------------|
| `get_weather` | Weather forecast | city, date |
| `convert_currency` | Currency conversion | amount, from, to |
| `search_hotels` | Hotel search with budget | city, dates, max_price |
| `get_attractions` | Local sights/restaurants | city, category |

In [ ]:
# Inspect the tools — look at their signatures and docstrings
import inspect

for name, func in TOOL_REGISTRY.items():
    sig = inspect.signature(func)
    print(f"\n{'='*60}")
    print(f"{name}{sig}")
    print(f"{'='*60}")
    print(inspect.getdoc(func))

In [ ]:
# Try calling them directly — they're just functions
print("Weather in Rome:")
print(json.dumps(TOOL_REGISTRY["get_weather"]("Rome", "2026-03-15"), indent=2))

print("\nHotels in Barcelona:")
print(json.dumps(TOOL_REGISTRY["search_hotels"]("Barcelona", "2026-03-20", "2026-03-23", 150), indent=2))

In [ ]:
# The registry is just a dict: name -> function
# This is the bridge between the LLM (which outputs a name) and your code
print("Registry:")
for name, func in TOOL_REGISTRY.items():
    print(f"  {name!r} -> {func.__name__}()")

## 2. Manual Tool Calling (no API support)

Before APIs had built-in tool calling, the only option was:
1. Describe your tools in the system prompt
2. Ask the model to output JSON when it wants to call a tool
3. Parse the JSON from the model's text response
4. Execute the function and feed the result back

Let's try this approach — and see where it breaks.

In [ ]:
MANUAL_SYSTEM_PROMPT = """You are a travel planning assistant with access to these tools:

1. get_weather(city: str, date: str) - Get weather forecast for a city on a date
2. convert_currency(amount: float, from_currency: str, to_currency: str) - Convert between currencies
3. search_hotels(city: str, checkin: str, checkout: str, max_price: int) - Search hotels within budget
4. get_attractions(city: str, category: str) - Get local attractions (landmarks/museums/restaurants/parks)

When you need to call a tool, output EXACTLY this JSON format on its own line:
{"tool": "function_name", "args": {"param1": "value1", "param2": "value2"}}

Output ONLY the JSON, nothing else. Do not wrap it in markdown code blocks.
After receiving the tool result, provide a natural language answer."""


def extract_json_from_text(text: str) -> dict | None:
    """Try to extract a JSON tool call from model output."""
    # Try the raw text first
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    # Try to find JSON inside markdown code blocks
    if "```" in text:
        start = text.index("```") + 3
        if text[start:].startswith("json"):
            start += 4
        end = text.index("```", start)
        try:
            return json.loads(text[start:end].strip())
        except json.JSONDecodeError:
            pass
    return None


print("Manual tool-calling system prompt loaded.")

In [ ]:
# Simple query — this usually works
user_query = "What's the weather in Rome on March 15, 2026?"

print(f"User: {user_query}\n")

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": MANUAL_SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ],
    temperature=0,
)
raw_output = r.choices[0].message.content
print(f"Model output:\n{raw_output}\n")

# Try to parse it
parsed = extract_json_from_text(raw_output)
if parsed and "tool" in parsed:
    print(f"Parsed tool call: {parsed['tool']}({parsed['args']})")
    result = execute_tool(parsed["tool"], parsed["args"])
    print(f"Tool result: {result}\n")

    # Send the result back to get a natural language answer
    r2 = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": MANUAL_SYSTEM_PROMPT},
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": raw_output},
            {"role": "user", "content": f"Tool result: {result}"},
        ],
        temperature=0,
    )
    print(f"Final answer: {r2.choices[0].message.content}")
else:
    print("Could not parse a tool call from the model output!")

### Where manual parsing breaks

The simple case works, but real models are unpredictable in their formatting. Let's stress-test with different models and temperatures.

In [ ]:
# Run the same query 5 times with temperature=0.7 and check parse success
test_queries = [
    "What's the weather in Rome on March 15, 2026?",
    "Find me a hotel in Barcelona under 120 euros, checking in March 20 and out March 23",
    "How much is 500 USD in Japanese Yen?",
    "What are some good restaurants in Tokyo?",
    "I need the weather AND hotels in Paris for next week",  # multi-tool — tricky!
]

for model_name in [MODEL, MODEL_STRONG]:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}, temperature=0.7")
    print(f"{'='*60}")

    successes, failures = 0, 0
    for q in test_queries:
        r = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": MANUAL_SYSTEM_PROMPT},
                {"role": "user", "content": q},
            ],
            temperature=0.7,
        )
        raw = r.choices[0].message.content
        parsed = extract_json_from_text(raw)
        ok = parsed is not None and "tool" in parsed
        if ok:
            successes += 1
        else:
            failures += 1
        status = "OK" if ok else "FAIL"
        print(f"  [{status}] {q[:50]}...")
        if not ok:
            print(f"         Raw output: {raw[:120]}...")

    print(f"  Score: {successes}/{successes + failures} parsed successfully")

**Common failure modes** you'll see:
- Model wraps JSON in ` ```json ` markdown blocks
- Model adds conversational text before/after the JSON
- Model uses slightly different field names (`"function"` instead of `"tool"`)
- Model tries to call multiple tools in one response but invents a format
- Model hallucinates tool names that don't exist

**Takeaway:** Manual parsing *works*, but you're fighting the model's formatting tendencies. Every edge case means more parsing code, more bugs, more brittleness.

## 3. API-Level Tool Calling

The OpenAI API has built-in support for tool calling. Instead of asking the model to output JSON text, you:
1. **Register** your tools with JSON schemas
2. The model returns structured `tool_calls` objects — no parsing needed
3. You execute the function, send the result back as a `tool` message

Let's see the same tools, but using the API properly.

In [ ]:
# Get the OpenAI tool schemas generated from our docstrings
tool_schemas = get_openai_tool_schemas()

print(f"{len(tool_schemas)} tool schemas generated:\n")
for ts in tool_schemas:
    print(json.dumps(ts, indent=2))
    print()

In [ ]:
# Same query, now using API tool calling
user_query = "What's the weather in Rome on March 15, 2026?"
print(f"User: {user_query}\n")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": user_query}],
    tools=tool_schemas,
    tool_choice="auto",
)

message = response.choices[0].message
print(f"Model wants to call tools: {message.tool_calls is not None}")

if message.tool_calls:
    for tc in message.tool_calls:
        print(f"\n  Tool: {tc.function.name}")
        print(f"  Args: {tc.function.arguments}")
        print(f"  ID:   {tc.id}")

        # Execute the tool
        result = execute_tool(tc.function.name, json.loads(tc.function.arguments))
        print(f"  Result: {result}")

In [ ]:
# Complete the round trip: send tool results back, get final answer
messages = [
    {"role": "user", "content": user_query},
    message,  # the assistant message with tool_calls
]

# Add tool results for each call
for tc in message.tool_calls:
    result = execute_tool(tc.function.name, json.loads(tc.function.arguments))
    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": result,
    })

# Get the final answer
r2 = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tool_schemas,
)

print(f"Final answer:\n{r2.choices[0].message.content}")

### Manual vs API: side-by-side comparison

| | Manual (Section 2) | API Tool Calling (Section 3) |
|---|---|---|
| **Format** | Hope the model outputs valid JSON | Guaranteed structured `tool_calls` objects |
| **Parsing** | Custom regex/JSON extraction code | `message.tool_calls[0].function.name` |
| **Multi-tool** | Model invents ad-hoc array format | API returns a list of tool_calls |
| **Reliability** | Breaks with temperature, model changes | Consistent across models and settings |
| **Code** | ~30 lines of fragile parsing | ~5 lines of structured access |

## 4. The Tool-Calling Loop

Real conversations often need **multiple tool calls** before the model can answer. The core pattern:

```
while True:
    response = LLM(messages, tools)
    if response has tool_calls:
        execute each tool
        append results to messages
    else:
        return response.text  # model is done
```

The **LLM decides** when it has enough information to answer.

In [ ]:
from prompt_templates import render_prompt


def tool_loop(
    user_message: str,
    *,
    model: str = MODEL,
    tool_schemas: list[dict] = None,
    execute_fn=execute_tool,
    system_prompt: str = None,
    max_iterations: int = 10,
    verbose: bool = True,
) -> str:
    """Run a tool-calling loop until the model produces a text response."""
    if tool_schemas is None:
        tool_schemas = get_openai_tool_schemas()

    if system_prompt is None:
        system_prompt, _ = render_prompt(
            "tool_system.j2",
            tools=tool_schemas,
            clarification_mode=False,
            user_message="",
        )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    total_prompt_tokens = 0
    total_completion_tokens = 0

    for iteration in range(1, max_iterations + 1):
        start = time.time()
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tool_schemas,
            tool_choice="auto",
        )
        elapsed = time.time() - start
        total_prompt_tokens += response.usage.prompt_tokens
        total_completion_tokens += response.usage.completion_tokens

        msg = response.choices[0].message

        if not msg.tool_calls:
            # Model is done — it produced a text answer
            if verbose:
                print(f"\n  Iteration {iteration}: MODEL ANSWERS ({elapsed:.1f}s)")
                print(f"  Total tokens — prompt: {total_prompt_tokens:,}, completion: {total_completion_tokens:,}")
            return msg.content

        # Model wants to call tools
        if verbose:
            print(f"\n  Iteration {iteration}: {len(msg.tool_calls)} tool call(s) ({elapsed:.1f}s)")

        messages.append(msg)

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"    -> {tc.function.name}({args})")

            try:
                result = execute_fn(tc.function.name, args)
            except Exception as e:
                result = json.dumps({"error": str(e)})

            if verbose:
                print(f"       Result: {result[:120]}{'...' if len(result) > 120 else ''}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return "[Max iterations reached without a final answer]"


print("tool_loop() defined.")

In [ ]:
# Simple query — should take 1 tool call
print("=" * 60)
print("Query: What's the weather in Rome on March 15?")
print("=" * 60)

answer = tool_loop("What's the weather in Rome on March 15, 2026?")
print(f"\nAnswer:\n{answer}")

In [ ]:
# Multi-step query — the model needs several tools
print("=" * 60)
print("Query: Plan a trip to Barcelona")
print("=" * 60)

answer = tool_loop(
    "I'm planning a trip to Barcelona next week (March 16-20, 2026). "
    "What's the weather like, find me a hotel under 150 EUR, "
    "and what are the must-see attractions?"
)
print(f"\nAnswer:\n{answer}")

In [ ]:
# Compare how gpt-4.1-mini and gpt-5.2 handle the same multi-step query
multi_query = (
    "I want to visit either Rome or Tokyo next week (March 16-20, 2026). "
    "Compare the weather in both cities, find hotels under 180 EUR in each, "
    "and show me restaurant recommendations for the better-weather city."
)

for model_name in [MODEL, MODEL_STRONG]:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    answer = tool_loop(multi_query, model=model_name)
    print(f"\nAnswer:\n{answer}")

## 5. When to Answer vs. When to Call Another Tool

The model **decides** at each step: do I have enough information, or do I need another tool call?

Let's trace this decision-making by comparing simple and complex queries.

In [ ]:
# Simple: 1 tool call, then done
print("=== Simple query ===")
answer = tool_loop("What's the weather in Rome on March 18, 2026?")
print(f"\nAnswer: {answer}\n")

print("\n" + "=" * 60 + "\n")

# Complex: multiple tool calls needed, model synthesizes at the end
print("=== Complex query ===")
answer = tool_loop(
    "Compare weather in Rome and Barcelona for March 18, 2026, "
    "and find the cheaper hotel option (check-in Mar 18, check-out Mar 21) in each city."
)
print(f"\nAnswer: {answer}")

In [ ]:
# What happens when a tool returns an error?
def execute_with_errors(name: str, arguments: dict) -> str:
    """Wrapper that simulates a failure for one specific call."""
    if name == "search_hotels" and arguments.get("city") == "Atlantis":
        return json.dumps({"error": "City not found: 'Atlantis'. No hotel data available."})
    return execute_tool(name, arguments)

print("=== Query with a tool error ===")
answer = tool_loop(
    "Find me hotels in Atlantis for March 20-23, 2026 under 200 EUR.",
    execute_fn=execute_with_errors,
)
print(f"\nAnswer: {answer}")

**Key insight:** The model uses tool results to decide its next move:
- Got all the info needed? → Produce a final text answer
- Need more data to complete the comparison? → Call another tool
- Got an error? → Explain to the user or try an alternative

## 6. Tool Naming and Documentation Matter

The model picks which tool to call based on the tool's **name**, **description**, and **parameter names**. What happens when these are bad?

We have `tools_bad.py` — same 4 functions, same logic, but:
- Generic names: `get_data`, `do_lookup`, `fetch`, `query`
- No parameter descriptions
- Cryptic param names: `p1`, `p2`, `x`, `y`, `z` instead of `city`, `date`

In [ ]:
import tools_bad

good_schemas = get_openai_tool_schemas()
bad_schemas = tools_bad.get_openai_tool_schemas()

# Compare a schema side by side
print("=== GOOD tool schema (get_weather) ===")
print(json.dumps(good_schemas[0], indent=2))
print("\n=== BAD tool schema (get_data) ===")
print(json.dumps(bad_schemas[0], indent=2))

In [ ]:
# Test both tool sets on the same queries
test_queries = [
    "What's the weather in Rome on March 15, 2026?",
    "How much is 200 EUR in GBP?",
    "Find hotels in Tokyo, checking in March 20, out March 23, under 150 EUR.",
    "What museums are there in Barcelona?",
]

# Expected tool for each query
expected_good = ["get_weather", "convert_currency", "search_hotels", "get_attractions"]
expected_bad = ["get_data", "do_lookup", "fetch", "query"]


def test_tool_selection(schemas, expected_tools, label):
    """Run queries and check if the model picks the right tool."""
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    correct = 0
    for q, expected in zip(test_queries, expected_tools):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": q}],
            tools=schemas,
            tool_choice="auto",
        )
        msg = r.choices[0].message
        if msg.tool_calls:
            called = msg.tool_calls[0].function.name
            args = msg.tool_calls[0].function.arguments
            ok = called == expected
            correct += ok
            status = "OK" if ok else "WRONG"
            print(f"  [{status}] {q[:50]}")
            print(f"          Called: {called}({args})")
            if not ok:
                print(f"          Expected: {expected}")
        else:
            print(f"  [MISS] {q[:50]}")
            print(f"          Model didn't call any tool!")
    print(f"  Score: {correct}/{len(test_queries)}")


test_tool_selection(good_schemas, expected_good, "GOOD tool definitions")
test_tool_selection(bad_schemas, expected_bad, "BAD tool definitions")

In [ ]:
# Repeat with gpt-5.2 — a stronger model might cope better with bad naming
def test_tool_selection_model(schemas, expected_tools, label, model):
    print(f"\n{'='*60}")
    print(f"{label} — {model}")
    print(f"{'='*60}")
    correct = 0
    for q, expected in zip(test_queries, expected_tools):
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": q}],
            tools=schemas,
            tool_choice="auto",
        )
        msg = r.choices[0].message
        if msg.tool_calls:
            called = msg.tool_calls[0].function.name
            args = msg.tool_calls[0].function.arguments
            ok = called == expected
            correct += ok
            status = "OK" if ok else "WRONG"
            print(f"  [{status}] {q[:50]}")
            print(f"          Called: {called}({args})")
        else:
            print(f"  [MISS] {q[:50]}")
    print(f"  Score: {correct}/{len(test_queries)}")

test_tool_selection_model(bad_schemas, expected_bad, "BAD tools", MODEL)
test_tool_selection_model(bad_schemas, expected_bad, "BAD tools", MODEL_STRONG)

### Best Practices for Tool Definitions

| Practice | Why it matters |
|----------|---------------|
| **Descriptive function names** | `search_hotels` vs `fetch` — the model needs to match intent to tool |
| **Parameter names that explain themselves** | `city` vs `p1` — the model needs to map user concepts to params |
| **Rich descriptions** | Tells the model *when* to use the tool and *what* each param expects |
| **Type constraints** | `max_price: int` vs `d` — helps the model pass valid values |
| **Examples in descriptions** | Reduces ambiguity for edge cases |
| **Consistent naming conventions** | If all tools follow a pattern, the model learns the pattern |

## 7. Asking Clarification Questions (Advanced Pattern)

What should the model do when the user's request is **ambiguous or incomplete**?

- Default behavior: guess and fill in parameters (often wrong)
- Better: **ask the user** for missing information

This isn't a special API feature — it's a system prompt instruction. The model chooses to produce text (a question) instead of a tool call.

In [ ]:
# First: default behavior (no clarification instruction) — model guesses
print("=== WITHOUT clarification instructions ===")

system_no_clarify, _ = render_prompt(
    "tool_system.j2",
    tools=tool_schemas,
    clarification_mode=False,
    user_message="",
)

vague_queries = [
    "Find me a hotel.",
    "What's the weather?",
    "Compare hotels.",
]

for q in vague_queries:
    print(f"\nUser: {q}")
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_no_clarify},
            {"role": "user", "content": q},
        ],
        tools=tool_schemas,
        tool_choice="auto",
    )
    msg = r.choices[0].message
    if msg.tool_calls:
        tc = msg.tool_calls[0]
        print(f"  Model GUESSED and called: {tc.function.name}({tc.function.arguments})")
    else:
        print(f"  Model responded: {msg.content[:150]}")

In [ ]:
# Now: with clarification instructions — model asks instead of guessing
print("=== WITH clarification instructions ===")

system_with_clarify, _ = render_prompt(
    "tool_system.j2",
    tools=tool_schemas,
    clarification_mode=True,
    user_message="",
)

for q in vague_queries:
    print(f"\nUser: {q}")
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_with_clarify},
            {"role": "user", "content": q},
        ],
        tools=tool_schemas,
        tool_choice="auto",
    )
    msg = r.choices[0].message
    if msg.tool_calls:
        tc = msg.tool_calls[0]
        print(f"  Model GUESSED and called: {tc.function.name}({tc.function.arguments})")
    else:
        print(f"  Model ASKED: {msg.content[:200]}")

In [ ]:
# Compare models: does a stronger model ask better clarification questions?
print("=== Clarification behavior: gpt-4.1-mini vs gpt-5.2 ===")

for model_name in [MODEL, MODEL_STRONG]:
    print(f"\n--- {model_name} ---")
    for q in vague_queries:
        r = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_with_clarify},
                {"role": "user", "content": q},
            ],
            tools=tool_schemas,
            tool_choice="auto",
        )
        msg = r.choices[0].message
        if msg.tool_calls:
            print(f"  User: {q}")
            print(f"  -> GUESSED: {msg.tool_calls[0].function.name}")
        else:
            print(f"  User: {q}")
            print(f"  -> ASKED: {msg.content[:150]}")

In [ ]:
# Full clarification flow: vague query -> clarification -> complete query -> answer
print("=== Full clarification conversation ===")

messages = [
    {"role": "system", "content": system_with_clarify},
    {"role": "user", "content": "Find me a hotel."},
]

# Turn 1: model should ask for details
r1 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=tool_schemas, tool_choice="auto"
)
msg1 = r1.choices[0].message
print(f"User: Find me a hotel.")
print(f"Assistant: {msg1.content}\n")

# Turn 2: user provides the missing info
messages.append(msg1)
messages.append({"role": "user", "content": "Barcelona, March 20-23, under 130 EUR per night."})

# Turn 2 response: model should now call the tool
r2 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=tool_schemas, tool_choice="auto"
)
msg2 = r2.choices[0].message
print(f"User: Barcelona, March 20-23, under 130 EUR per night.")

if msg2.tool_calls:
    tc = msg2.tool_calls[0]
    print(f"Assistant calls: {tc.function.name}({tc.function.arguments})")

    result = execute_tool(tc.function.name, json.loads(tc.function.arguments))
    messages.append(msg2)
    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    r3 = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tool_schemas
    )
    print(f"\nAssistant: {r3.choices[0].message.content}")
else:
    print(f"Assistant: {msg2.content}")

### Key insight

**Asking back is itself a "tool"** — the model's ability to produce text is its most flexible capability. A tool call requires exact parameters; a clarification question can handle any ambiguity.

The model has three options at each step:
1. **Call a tool** — if it knows which tool and has all the parameters
2. **Ask a question** — if information is missing or ambiguous
3. **Answer directly** — if it already has enough information

The system prompt determines which of these behaviors the model prefers. Without clarification instructions, models tend to guess rather than ask — which can lead to wrong tool calls with hallucinated parameters.